# Michigan Ave route identification

Which candidate patterns **meaningfully serve** the downtown Michigan Ave corridor (Roosevelt → Delaware)?

1. **Segment geometry** — longest contiguous corridor stop run from cached Bus Tracker `getpatterns` (same as stop discovery).
2. **Segment timing** — median trip time across that run from `data/mansueto/trips_{pid}_last30d.parquet` (cache via `scripts/cache_dlsd_parquets.py --source michigan --days 30`).
3. **Eligibility** — vault `michigan-ave-bus-lane-segment`: keep patterns with **median ≥ 5 minutes** (interpolation ~5 min between GPS anchors).

Outputs: summary table, histogram, `michigan_ave_kept_pids.json`.

In [ ]:
%matplotlib inline

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

_nb = Path.cwd() if (Path.cwd() / "michigan_ave_segment.py").is_file() else Path.cwd() / "notebooks"
if str(_nb) not in sys.path:
    sys.path.insert(0, str(_nb))

from michigan_ave_segment import (
    DEFAULT_MIN_ENTRY_EXIT_SHARE,
    DEFAULT_MIN_MEDIAN_MINUTES,
    apply_median_threshold,
    build_corridor_segments,
    build_endpoint_coverage_summary,
    build_pid_timing_summary,
    candidate_pids,
    load_corridor_stpids,
)

REPO_ROOT = _nb.parent if _nb.name == "notebooks" else _nb
KEPT_PATH = _nb / "michigan_ave_kept_pids.json"
_style = REPO_ROOT / "notebooks" / "cta_analysis.mplstyle"
if _style.is_file():
    plt.style.use(_style)

CACHE_VARIANT = "last30d"
MIN_MEDIAN = DEFAULT_MIN_MEDIAN_MINUTES

In [ ]:
corridor_stpids = load_corridor_stpids()
print(f"Corridor stops: {len(corridor_stpids)}")
print(f"Catalog PIDs (rt_to_pid): {len(candidate_pids())}")

segments = build_corridor_segments(use_cache=True)
print(f"PIDs with a corridor segment (≥2 stops): {len(segments)}")

In [ ]:
summary = build_pid_timing_summary(segments, cache_variant=CACHE_VARIANT)
summary = summary.sort_values("median_minutes", ascending=False, na_position="last")
print(f"PIDs with timed trips in cache: {len(summary)}")
display(summary)

## Pattern endpoint coverage

For each corridor segment, count trips that hit the pattern **entry** and **exit** (`longest_corridor_run` endpoints). Flag when **fewer than 50%** of entry trips also hit the exit — often a Bus Tracker stop the published trips never visit (e.g. 148 SB PID 6402: Delaware observed, Monroe never). Do not impute missing stops; see vault `processed-parquet-verification-checklist`.

In [ ]:
coverage = build_endpoint_coverage_summary(
    segments,
    cache_variant=CACHE_VARIANT,
    min_entry_exit_share=DEFAULT_MIN_ENTRY_EXIT_SHARE,
)
print(f"PIDs with coverage stats: {len(coverage)}")
flagged = coverage.loc[coverage["low_entry_exit_coverage"]].copy()
print(
    f"Flagged (share entry→exit < {DEFAULT_MIN_ENTRY_EXIT_SHARE:.0%}): {len(flagged)}"
)

display_cols = [
    "route",
    "pid",
    "rtdir",
    "n_trips_entry",
    "n_trips_exit",
    "n_trips_both",
    "share_entry_with_exit",
    "entry_stpnm",
    "exit_stpnm",
    "mode_last_corridor_stpnm",
    "mode_last_corridor_n_trips",
]
display(coverage[display_cols])

if not flagged.empty:
    print("\nLow entry→exit coverage (worst first):")
    display(flagged[display_cols])

In [ ]:
plotted = summary.dropna(subset=["median_minutes"])
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(plotted["median_minutes"], bins=20, edgecolor="white")
ax.axvline(MIN_MEDIAN, color="C3", linestyle="--", label=f"threshold = {MIN_MEDIAN} min")
ax.set_xlabel("Median corridor time per PID (minutes)")
ax.set_ylabel("PID count")
ax.set_title("Michigan Ave segment — median trip time by pattern")
ax.legend()
plt.tight_layout()
plt.show()

below = int((plotted["median_minutes"] < MIN_MEDIAN).sum())
print(
    f"{below} PIDs below {MIN_MEDIAN} min median — likely clip the corridor or sit in one ping interval."
)

In [ ]:
kept = apply_median_threshold(summary, min_median_minutes=MIN_MEDIAN)
routes_kept = sorted(kept["route"].unique(), key=lambda r: (r.isdigit(), r))
print(f"Kept {len(kept)} PIDs across {len(routes_kept)} routes: {', '.join(routes_kept)}")
display(kept[["route", "pid", "rtdir", "median_minutes", "n_trips", "entry_stpnm", "exit_stpnm"]])

payload = {
    "min_median_minutes": MIN_MEDIAN,
    "cache_variant": CACHE_VARIANT,
    "routes_identified": routes_kept,
    "pids": kept.to_dict(orient="records"),
}
KEPT_PATH.write_text(json.dumps(payload, indent=2) + "\n")
print(f"Wrote {KEPT_PATH.name}")